# Hematuria ICD codes + time stamps (SPARK in UKB RAP)

## Setting up environment

In [20]:
### Loading libraries
import dxpy
import dxdata
import pandas as pd
import pyspark
import pyspark.sql.functions as F
from functools import reduce
from operator import or_

In [2]:
### Spark initialization (Done only once; do not rerun this cell unless you select Kernel -> Restart kernel).
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

In [4]:
### Automatically discover dispensed dataset ID and load the dataset 
dispensed_dataset_id = dxpy.find_one_data_object(typename='Dataset', name='app*.dataset', folder='/', name_mode='glob')['id']
dataset = dxdata.load_dataset(id=dispensed_dataset_id)

### selecting participants entity
participant = dataset['participant']

## Phenotype fields to retrieve
The list of fields which I want to retreive from the dataset to be later used in phenotypes and covariates:

| Field | Title |
|------|------|
| eid | id of a participant |
| p41270 | Diagnoses - ICD10 |
| p41271 | Diagnoses - ICD9 |
| p41281_a0 - p41281_a46 | Time stamp ICD9 |
| p41280_a0 - p41280_a258 | Time stamp ICD10 |

In [ ]:
### Reading in the fields codes
field_names_ICD9 = ['eid', 'p41271'] + [f'p41281_a{i}' for i in range(47)]

field_names_ICD10 = ['eid', 'p41270'] + [f'p41280_a{i}' for i in range(259)]

In [6]:
### Retrieving information for these selected fields from UKB
df_ICD9 = participant.retrieve_fields(names=field_names_ICD9, engine=dxdata.connect())
df_ICD10 = participant.retrieve_fields(names=field_names_ICD10, engine=dxdata.connect())

In [ ]:
### see the header of the spark table in pandas
# df_ICD9.limit(5).toPandas()

## Phecode 593 phenotype

In [35]:
codes_ICD9 = [
    # ICD-9
    "599.7", "599.70", "599.71"
]

codes_ICD10 = [
    # ICD-10
    "R31", "R31.9", "R31.0"
]

columns_ICD9 = [
    # ICD-9
    'p41271']

columns_ICD10 = [
    # ICD-10
    "p41270"]

### Convert list of codes to array of literals
code_array_ICD9 = F.array(*[F.lit(code) for code in codes_ICD9])
code_array_ICD10 = F.array(*[F.lit(code) for code in codes_ICD10])

### Build OR condition across all columns
condition_ICD9 = reduce(or_, [F.arrays_overlap(F.col(c), code_array_ICD9) for c in columns_ICD9])
condition_ICD10 = reduce(or_, [F.arrays_overlap(F.col(c), code_array_ICD9) for c in columns_ICD10])

### Apply the filter and add a new binary column for hematuria
filtered_df_ICD9 = df_ICD9.withColumn("hematuria", F.when(condition_ICD9, 1).otherwise(0))
filtered_df_ICD10 = df_ICD10.withColumn("hematuria", F.when(condition_ICD10, 1).otherwise(0))

## Output

In [38]:
filtered_df_ICD9.toPandas().to_csv('hematuria_ICD9_timestamp.txt', sep='\t', index=False)
filtered_df_ICD10.toPandas().to_csv('hematuria_ICD10_timestamp.txt', sep='\t', index=False)

In [ ]:
%%bash
dx upload hematuria_ICD9_timestamp.txt --dest hematuria_ICD9_timestamp.txt
dx upload hematuria_ICD10_timestamp.txt --dest hematuria_ICD10_timestamp.txt